In [5]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Adathalmaz beolvasása
df = pd.read_csv('Fifa 23 Players Data.csv')

# Sorok és oszlopok száma
print(f"Sorok száma: {df.shape[0]}, Oszlopok száma: {df.shape[1]}")

# Első 30 oszlopnév kiíratása ellenőrzéshez
print(df.columns.tolist()[:30])

Sorok száma: 18539, Oszlopok száma: 89
['Known As', 'Full Name', 'Overall', 'Potential', 'Value(in Euro)', 'Positions Played', 'Best Position', 'Nationality', 'Image Link', 'Age', 'Height(in cm)', 'Weight(in kg)', 'TotalStats', 'BaseStats', 'Club Name', 'Wage(in Euro)', 'Release Clause', 'Club Position', 'Contract Until', 'Club Jersey Number', 'Joined On', 'On Loan', 'Preferred Foot', 'Weak Foot Rating', 'Skill Moves', 'International Reputation', 'National Team Name', 'National Team Image Link', 'National Team Position', 'National Team Jersey Number']


In [3]:
%pip install numpy pandas plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# 1. KÉRDÉS: Kor és átlagos Overall képesség kapcsolata
age_overall = df.groupby('Age')['Overall'].agg(['mean', 'count']).reset_index()
# Csak azokat a korcsoportokat nézzük, ahol van releváns számú játékos (pl. legalább 30 fő)
age_overall = age_overall[age_overall['count'] >= 30]

fig1 = px.line(
    age_overall, 
    x='Age', 
    y='mean', 
    title='Játékosok átlagos összértéke (Overall) életkor szerint',
    labels={'Age': 'Életkor', 'mean': 'Átlagos Overall'},
    markers=True
)
fig1.show()


In [13]:
# 2. KÉRDÉS: Érték (Value) és Fizetés (Wage) kapcsolata
# Kiszűrjük a 0 eurós értékeket a logaritmáláshoz
market_df = df[(df['Value(in Euro)'] > 0) & (df['Wage(in Euro)'] > 0)].copy()

# NumPy számítás: logaritmikus transzformáció és korreláció
log_value = np.log10(market_df['Value(in Euro)'])
log_wage = np.log10(market_df['Wage(in Euro)'])
corr = np.corrcoef(log_value, log_wage)[0, 1]
print(f"NumPy korrelációs együttható az érték és fizetés logaritmusa között: {corr:.3f}")

fig2 = px.scatter(
    market_df,
    x='Value(in Euro)',
    y='Wage(in Euro)',
    color='Overall',
    hover_name='Known As',
    log_x=True,
    log_y=True,
    title='Játékos piaci értéke vs. Heti fizetése (Logaritmikus skálán)',
    labels={'Value(in Euro)': 'Piaci érték (€)', 'Wage(in Euro)': 'Heti fizetés (€)'}
)
fig2.show()


NumPy korrelációs együttható az érték és fizetés logaritmusa között: 0.745


In [14]:
# 3. KÉRDÉS: Legértékesebb pozíciók
pos_value = df.groupby('Best Position')['Value(in Euro)'].median().sort_values(ascending=False).reset_index()

fig3 = px.bar(
    pos_value,
    x='Best Position',
    y='Value(in Euro)',
    title='Medián piaci érték pozíciónként',
    labels={'Best Position': 'Legjobb pozíció', 'Value(in Euro)': 'Medián érték (€)'},
    color='Value(in Euro)',
    color_continuous_scale='Viridis'
)
fig3.show()


In [15]:
# 4. KÉRDÉS: Korrelációs vizsgálat NumPy-val a kulcs statisztikák között
cols_to_check = ['Overall', 'Potential', 'Value(in Euro)', 'Wage(in Euro)', 'Age']
sub_data = df[cols_to_check].dropna()

# NumPy korrelációs mátrix kiszámítása
corr_matrix = np.corrcoef(sub_data.values, rowvar=False)

fig4 = px.imshow(
    corr_matrix,
    x=cols_to_check,
    y=cols_to_check,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='NumPy Korrelációs Mátrix az Alapmutatók Között'
)
fig4.show()


In [16]:
# 5. KÉRDÉS: Top nemzetek elit játékosai
top_players = df[df['Overall'] >= 85]
top_nations = top_players['Nationality'].value_counts().head(10).reset_index()
top_nations.columns = ['Nationality', 'Count']

fig5 = px.bar(
    top_nations,
    x='Nationality',
    y='Count',
    title='Top nemzetek az elit (85+ Overall) játékosok száma alapján',
    labels={'Nationality': 'Nemzetiség', 'Count': 'Elit játékosok száma'},
    color='Count',
    color_continuous_scale='Blues'
)
fig5.show()


In [11]:
# nbformat telepítése a megjelenítéshez
%pip install nbformat

# Plotly renderer beállítása közvetlenül a notebookhoz
import plotly.io as pio
pio.renderers.default = "notebook_connected"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
